In [2]:
import os
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

In [3]:
SPLITS = "/kaggle/input/datasets/iwmm10/chestxray-capstone-splits"
BASE   = f"{SPLITS}/capstone_splits"
NIH    = "/kaggle/input/datasets/nih-chest-xrays/data"
OUT    = "/kaggle/working"

SPLIT_DIR = f"{BASE}/splits"
INDEX_JSON = f"{SPLITS}/image_index.json"

CKPT = "/kaggle/input/datasets/iwmm10/baseline/resnet50_baseline.pt"   # <-- edit this

IMG_COL, PID_COL = "Image Index", "Patient ID"

LABELS = [
    "Atelectasis", "Consolidation", "Infiltration", "Pneumothorax",
    "Edema", "Emphysema", "Fibrosis", "Effusion",
    "Pneumonia", "Pleural_Thickening", "Cardiomegaly", "Nodule",
    "Mass", "Hernia",
]

IMG_SIZE, SEED = 224, 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(INDEX_JSON) as f:
    index = json.load(f)

def image_path(fn):
    return f"{NIH}/{index[fn]}/images/{fn}"

print("device:", device, "|", f"{len(index):,} images indexed")

device: cuda | 112,120 images indexed


In [4]:
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ChestXrayDataset(Dataset):
    def __init__(self, df, transform):
        self.files  = df[IMG_COL].values
        self.labels = df[LABELS].values.astype("float32")
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(image_path(self.files[i])).convert("RGB")
        return self.transform(img), torch.from_numpy(self.labels[i])

val_df  = pd.read_csv(f"{SPLIT_DIR}/val.csv")
test_df = pd.read_csv(f"{SPLIT_DIR}/test.csv")

def make_loader(df):
    return DataLoader(ChestXrayDataset(df, eval_tf), batch_size=32,
                      shuffle=False, num_workers=4, pin_memory=True)

val_loader, test_loader = make_loader(val_df), make_loader(test_df)
print(f"val {len(val_df):,} | test {len(test_df):,}")

val 3,978 | test 3,904


In [5]:
def predict(checkpoint_path, loader):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(LABELS))

    sd = torch.load(checkpoint_path, map_location="cpu")

    # Project standard is 14 outputs — 'No Finding' is not a class
    out_dim = sd["fc.weight"].shape[0]
    if out_dim != len(LABELS):
        raise ValueError(f"Checkpoint has {out_dim} outputs, expected {len(LABELS)}")

    m.load_state_dict(sd)
    m.to(device).eval()

    probs, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            with torch.amp.autocast('cuda'):
                out = m(xb.to(device, non_blocking=True))
            probs.append(torch.sigmoid(out).float().cpu())
            targets.append(yb)

    return torch.cat(probs).numpy(), torch.cat(targets).numpy()

In [6]:
def bootstrap_ci(y_true, y_prob, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if y_true[idx].sum() == 0:
            continue
        scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(scores, [2.5, 97.5])


def report(p, t, name):
    print(f"\n{'='*62}\n{name}\n{'='*62}")
    print(f"{'label':<20}{'AUROC':>9}{'95% CI':>20}{'AP':>9}{'n_pos':>8}")

    rows, aucs = [], []
    for i, label in enumerate(LABELS):
        if t[:, i].sum() == 0:
            continue
        auc = roc_auc_score(t[:, i], p[:, i])
        ap  = average_precision_score(t[:, i], p[:, i])
        lo, hi = bootstrap_ci(t[:, i], p[:, i])
        aucs.append(auc)
        rows.append((label, auc, lo, hi, ap, int(t[:, i].sum())))
        print(f"{label:<20}{auc:>9.4f}   [{lo:.3f}, {hi:.3f}]{ap:>9.4f}{int(t[:, i].sum()):>8,}")

    macro = float(np.mean(aucs))
    print(f"{'-'*56}\n{'MACRO AUROC':<20}{macro:>9.4f}")
    return pd.DataFrame(rows, columns=["label","auroc","ci_lo","ci_hi","ap","n_pos"]), macro

In [7]:
p_test, t_test = predict(CKPT, test_loader)
df_res, macro_res = report(p_test, t_test, "ResNet50 — TEST")

df_res.to_csv(f"{OUT}/resnet50_test_results.csv", index=False)
print("\nsaved resnet50_test_results.csv")


ResNet50 — TEST
label                   AUROC              95% CI       AP   n_pos
Atelectasis            0.7375   [0.716, 0.759]   0.3569     612
Consolidation          0.7424   [0.716, 0.770]   0.2300     333
Infiltration           0.6962   [0.675, 0.717]   0.3656     838
Pneumothorax           0.8150   [0.792, 0.837]   0.3348     375
Edema                  0.9061   [0.891, 0.921]   0.4646     310
Emphysema              0.8839   [0.864, 0.903]   0.6008     380
Fibrosis               0.7837   [0.754, 0.812]   0.2126     212
Effusion               0.8240   [0.808, 0.841]   0.5639     770
Pneumonia              0.6725   [0.625, 0.715]   0.0930     163
Pleural_Thickening     0.7463   [0.716, 0.777]   0.2532     309
Cardiomegaly           0.8651   [0.841, 0.889]   0.5217     288
Nodule                 0.7173   [0.689, 0.743]   0.2513     386
Mass                   0.7268   [0.699, 0.754]   0.3033     374
Hernia                 0.8389   [0.751, 0.924]   0.2796      30
--------------------

In [8]:
p_val, t_val = predict(CKPT, val_loader)

grid = np.arange(0.05, 0.95, 0.01)
thresholds = {}
for i, label in enumerate(LABELS):
    f1s = [f1_score(t_val[:, i], (p_val[:, i] >= th).astype(int), zero_division=0)
           for th in grid]
    thresholds[label] = round(float(grid[int(np.argmax(f1s))]), 2)

print(f"{'label':<20}{'thresh':>8}{'F1@0.5':>9}{'F1@tuned':>10}")
f1_def, f1_tun = [], []
for i, label in enumerate(LABELS):
    a = f1_score(t_test[:, i], (p_test[:, i] >= 0.5).astype(int), zero_division=0)
    b = f1_score(t_test[:, i], (p_test[:, i] >= thresholds[label]).astype(int), zero_division=0)
    f1_def.append(a); f1_tun.append(b)
    print(f"{label:<20}{thresholds[label]:>8.2f}{a:>9.3f}{b:>10.3f}")

print(f"\n{'MACRO F1':<20}{'':>8}{np.mean(f1_def):>9.3f}{np.mean(f1_tun):>10.3f}")

with open(f"{OUT}/thresholds_resnet50.json", "w") as f:
    json.dump(thresholds, f, indent=2)

label                 thresh   F1@0.5  F1@tuned
Atelectasis             0.18    0.192     0.401
Consolidation           0.18    0.046     0.287
Infiltration            0.19    0.190     0.448
Pneumothorax            0.17    0.221     0.426
Edema                   0.19    0.432     0.486
Emphysema               0.34    0.539     0.576
Fibrosis                0.23    0.133     0.255
Effusion                0.24    0.428     0.549
Pneumonia               0.05    0.000     0.150
Pleural_Thickening      0.17    0.114     0.310
Cardiomegaly            0.21    0.468     0.531
Nodule                  0.16    0.069     0.279
Mass                    0.21    0.176     0.321
Hernia                  0.09    0.000     0.267

MACRO F1                        0.215     0.378


In [9]:
print(os.listdir(OUT))

['resnet50_test_results.csv', 'thresholds_resnet50.json', '.virtual_documents']
